# Data Quality — Configuration Validation Notebook

**Use this notebook to verify that LAKEHOUSE_NAME and SCHEMA_NAME are consistent across all operational data quality notebooks.**

Operational notebooks should load settings from `data_quality_config_notebook` and keep fallback defaults aligned.

This notebook detects misconfigurations early, so you don't encounter silent failures downstream.

## Expected Values

Expected values are sourced from shared config when available, then validated across all notebook files.

In [ ]:
# -- Expected Configuration Values --------------------------------------------
# Preferred: load expected values from shared config notebook.
try:
    ip = get_ipython()
    if ip is None:
        raise RuntimeError("IPython runtime not available")
    ip.run_line_magic("run", "data_quality_config_notebook")
    EXPECTED_LAKEHOUSE_NAME = LAKEHOUSE_NAME
    EXPECTED_SCHEMA_NAME = SCHEMA_NAME
except Exception:
    EXPECTED_LAKEHOUSE_NAME = "MyLakehouse"
    EXPECTED_SCHEMA_NAME = "data_quality"

# Operational notebooks that should use shared config values
NOTEBOOKS_TO_CHECK = [
    "data_quality_setup_notebook",
    "data_quality_add_checks_notebook",
    "data_quality_validation_job_notebook",
    "data_quality_delete_checks_notebook",
    "data_quality_rerun_check_notebook",
    "data_quality_rerun_failed_notebook",
    "power_bi_queries_notebook",
    "data_quality_smoke_test_notebook"
]

print(f"Expected LAKEHOUSE_NAME: {EXPECTED_LAKEHOUSE_NAME}")
print(f"Expected SCHEMA_NAME:    {EXPECTED_SCHEMA_NAME}")
print(f"\nChecking {len(NOTEBOOKS_TO_CHECK)} notebooks...")

## Validation Results

In [ ]:
import json
import re
from pathlib import Path
from IPython.display import display
import pandas as pd

# Strategy:
# 1) Preferred: scan local .ipynb sources (works in repo/local authoring)
# 2) Fallback: Fabric runtime validation when notebook files are not directly readable

note_dir = Path(".").resolve()


def scan_notebook_sources(notebook_dir, notebook_names):
    """Scan notebook source files for config values or shared-config usage."""
    results = []
    all_match = True

    for notebook_name in notebook_names:
        notebook_path = notebook_dir / f"{notebook_name}.ipynb"

        if not notebook_path.exists():
            results.append({
                "notebook": notebook_name,
                "lakehouse_name": "[FILE NOT FOUND]",
                "schema_name": "[FILE NOT FOUND]",
                "lakehouse_match": False,
                "schema_match": False,
                "status": "NOT FOUND"
            })
            all_match = False
            continue

        try:
            with open(notebook_path, "r", encoding="utf-8") as f:
                notebook_json = json.load(f)

            found_lakehouse = None
            found_schema = None
            uses_shared_config = False

            for cell in notebook_json.get("cells", []):
                if cell.get("cell_type") != "code":
                    continue

                source = "".join(cell.get("source", []))

                # Accept either notebook-based shared config or legacy python-module import patterns.
                if (
                    "run_line_magic(\"run\", \"data_quality_config_notebook\")" in source
                    or "run_line_magic('run', 'data_quality_config_notebook')" in source
                    or "%run data_quality_config_notebook" in source
                    or "from data_quality_config import" in source
                ):
                    uses_shared_config = True

                lakehouse_match = re.search(r'LAKEHOUSE_NAME\s*=\s*["\']([^"\']*)["\']', source)
                if lakehouse_match and not found_lakehouse:
                    found_lakehouse = lakehouse_match.group(1)

                schema_match = re.search(r'SCHEMA_NAME\s*=\s*["\']([^"\']*)["\']', source)
                if schema_match and not found_schema:
                    found_schema = schema_match.group(1)

            if uses_shared_config:
                found_lakehouse = EXPECTED_LAKEHOUSE_NAME
                found_schema = EXPECTED_SCHEMA_NAME
                status = "OK (shared config)"
                lakehouse_ok = True
                schema_ok = True
            else:
                lakehouse_ok = found_lakehouse == EXPECTED_LAKEHOUSE_NAME
                schema_ok = found_schema == EXPECTED_SCHEMA_NAME
                status = "OK" if (lakehouse_ok and schema_ok) else "MISMATCH"

            results.append({
                "notebook": notebook_name,
                "lakehouse_name": found_lakehouse or "[NOT FOUND]",
                "schema_name": found_schema or "[NOT FOUND]",
                "lakehouse_match": lakehouse_ok,
                "schema_match": schema_ok,
                "status": status
            })

            if not (lakehouse_ok and schema_ok):
                all_match = False

        except Exception as e:
            results.append({
                "notebook": notebook_name,
                "lakehouse_name": f"[ERROR: {str(e)[:40]}]",
                "schema_name": "[ERROR]",
                "lakehouse_match": False,
                "schema_match": False,
                "status": "ERROR"
            })
            all_match = False

    return results, all_match


def fabric_runtime_validation():
    """Validate that expected schema/tables are reachable in current runtime."""
    runtime_rows = []
    runtime_ok = True

    full_schema = f"{EXPECTED_LAKEHOUSE_NAME}.{EXPECTED_SCHEMA_NAME}"
    required_tables = ["check_registry", "validation_results"]

    # Schema reachability
    try:
        spark.sql(f"SHOW TABLES IN {full_schema}")
        runtime_rows.append({
            "Check": "Schema Reachability",
            "Target": full_schema,
            "Result": "OK",
            "Details": "Schema is reachable from this runtime"
        })
    except Exception as e:
        runtime_ok = False
        runtime_rows.append({
            "Check": "Schema Reachability",
            "Target": full_schema,
            "Result": "FAIL",
            "Details": str(e)[:180]
        })
        return runtime_rows, runtime_ok

    # Required tables exist and can be queried
    for tbl in required_tables:
        fqtn = f"{full_schema}.{tbl}"
        try:
            spark.sql(f"SELECT * FROM {fqtn} LIMIT 1")
            runtime_rows.append({
                "Check": "Table Readability",
                "Target": fqtn,
                "Result": "OK",
                "Details": "Table exists and is queryable"
            })
        except Exception as e:
            runtime_ok = False
            runtime_rows.append({
                "Check": "Table Readability",
                "Target": fqtn,
                "Result": "FAIL",
                "Details": str(e)[:180]
            })

    return runtime_rows, runtime_ok


existing_files = [
    name for name in NOTEBOOKS_TO_CHECK
    if (note_dir / f"{name}.ipynb").exists()
]

if len(existing_files) == len(NOTEBOOKS_TO_CHECK):
    print("Mode: Source scan (all notebook files found locally)")
    source_results, source_ok = scan_notebook_sources(note_dir, NOTEBOOKS_TO_CHECK)

    results_df = pd.DataFrame([
        {
            "Notebook": r["notebook"],
            "LAKEHOUSE_NAME": r["lakehouse_name"],
            "SCHEMA_NAME": r["schema_name"],
            "Status": r["status"]
        }
        for r in source_results
    ])

    display(results_df)

    print("\n" + "=" * 70)
    if source_ok:
        print("OK: All notebooks are configured consistently.")
        print(f"LAKEHOUSE_NAME = {EXPECTED_LAKEHOUSE_NAME}")
        print(f"SCHEMA_NAME    = {EXPECTED_SCHEMA_NAME}")
    else:
        print("FAIL: Configuration mismatches detected.")
        print("\nMismatched notebooks:")
        for r in source_results:
            if not (r["lakehouse_match"] and r["schema_match"]):
                print(f"- {r['notebook']}")
                if not r["lakehouse_match"]:
                    print(f"  LAKEHOUSE_NAME: {r['lakehouse_name']} (expected: {EXPECTED_LAKEHOUSE_NAME})")
                if not r["schema_match"]:
                    print(f"  SCHEMA_NAME: {r['schema_name']} (expected: {EXPECTED_SCHEMA_NAME})")
    print("=" * 70)

else:
    print("Mode: Fabric runtime fallback (notebook source files are not directly accessible)")
    print("Source-level cross-notebook consistency check is unavailable in this runtime.")
    print("Running operational validation against expected schema/tables instead...\n")

    runtime_rows, runtime_ok = fabric_runtime_validation()
    runtime_df = pd.DataFrame(runtime_rows)
    display(runtime_df)

    print("\n" + "=" * 70)
    if runtime_ok:
        print("OK: Runtime configuration is operational for expected lakehouse/schema.")
        print("Note: Source-level consistency across notebook files was not verifiable here.")
    else:
        print("FAIL: Runtime configuration issues found.")
        print("Fix lakehouse/schema values or table setup before running validation jobs.")
    print("=" * 70)